In [1]:
!pip install pytrec_eval sentence-transformers

  Preparing metadata (setup.py) ... done
  Created wheel for pytrec_eval: filename=pytrec_eval-0.5-cp312-cp312-linux_x86_64.whl size=309355 sha256=b67d1a26c7b06c4f154d79876cf49537048d1d9546e15a5d4239d3594a4edef7
  Stored in directory: /root/.cache/pip/wheels/c6/4a/9e/e17f9ea004e1c221bd0ff384732285211c4917b790d598ea51
Successfully built pytrec_eval


In [2]:
import json
import os
import pickle
import numpy as np
from collections import defaultdict
from numpy.linalg import norm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer
from google.colab import drive
from scipy.sparse import save_npz, load_npz
import pytrec_eval

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ==================== CẤU HÌNH ====================
CORPUS_PATH = "/content/drive/MyDrive/InformationRetrieval/Data/clean_corpus.jsonl"
QUERIES_PATH = "/content/drive/MyDrive/InformationRetrieval/Data/clean_queries.jsonl"
QRELS_PATH = "/content/drive/MyDrive/InformationRetrieval/Data/qrels/test.tsv"

SBERT_MODEL_NAME = "pritamdeka/S-PubMedBert-MS-MARCO"  # Biomedical SBERT model
SBERT_BATCH_SIZE = 64

OUTPUT_DIR_TFIDF = "/content/drive/MyDrive/InformationRetrieval/index_tfidf/"
OUTPUT_DIR_SBERT = "/content/drive/MyDrive/InformationRetrieval/index_sbert/"

In [4]:
# ==================== HÀM CHUNG ====================
def load_jsonl(path: str) -> dict:
    data = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                obj = json.loads(line)
                data[obj["_id"]] = obj["text"]
    return data

def load_qrels(path: str) -> dict:
    qrels = defaultdict(dict)
    with open(path, encoding="utf-8") as f:
        next(f, None)
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) < 3: continue
            q_id, c_id, score = parts[0], parts[1], int(parts[2])
            qrels[q_id][c_id] = score
    return dict(qrels)


In [5]:
# ==================== LOAD SBERT MODEL ====================
print(f"Đang load SBERT model: {SBERT_MODEL_NAME}...")
sbert_model = SentenceTransformer(SBERT_MODEL_NAME)
DIM_SBERT = sbert_model.get_sentence_embedding_dimension()
print(f"SBERT dim: {DIM_SBERT}")


Đang load SBERT model: pritamdeka/S-PubMedBert-MS-MARCO...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SBERT dim: 768


/tmp/ipykernel_1167/2769133732.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  DIM_SBERT = sbert_model.get_sentence_embedding_dimension()


In [6]:
# ==================== BUILD TF-IDF ====================
def build_tfidf_version(corpus):
    print("\n=== XÂY DỰNG TF-IDF THUẦN ===")
    doc_ids = list(corpus.keys())
    texts = list(corpus.values())

    tfidf = TfidfVectorizer(
        analyzer="word",
        token_pattern=r"(?u)\S+",
        min_df=1,
        max_df=0.9,
        sublinear_tf=False,
        norm='l2',
        use_idf=True,
        smooth_idf=True
    )

    D_tfidf = tfidf.fit_transform(texts)
    print(f"TF-IDF Vocab size: {len(tfidf.vocabulary_):,} terms")

    os.makedirs(OUTPUT_DIR_TFIDF, exist_ok=True)
    save_npz(os.path.join(OUTPUT_DIR_TFIDF, "doc_matrix.npz"), D_tfidf)
    with open(os.path.join(OUTPUT_DIR_TFIDF, "doc_ids.pkl"), "wb") as f:
        pickle.dump(doc_ids, f)
    with open(os.path.join(OUTPUT_DIR_TFIDF, "tfidf_vectorizer.pkl"), "wb") as f:
        pickle.dump(tfidf, f)

    print(f"Index TF-IDF đã lưu tại: {OUTPUT_DIR_TFIDF}")
    return D_tfidf, doc_ids, tfidf

In [7]:
# ==================== BUILD SBERT INDEX ====================
def build_sbert_index(corpus, sbert_model, output_dir, batch_size=64):
    print("\n=== XÂY DỰNG SBERT INDEX ===")
    doc_ids = list(corpus.keys())
    texts = list(corpus.values())
    N = len(doc_ids)

    print(f"Đang encode {N:,} documents bằng SBERT (batch_size={batch_size})...")
    D = sbert_model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,   # L2-normalize sẵn
        convert_to_numpy=True
    ).astype(np.float32)

    os.makedirs(output_dir, exist_ok=True)
    np.save(os.path.join(output_dir, "doc_matrix.npy"), D)
    with open(os.path.join(output_dir, "doc_ids.pkl"), "wb") as f:
        pickle.dump(doc_ids, f)

    print(f"SBERT index đã lưu tại: {output_dir}")
    return D, doc_ids


In [8]:
# ==================== COMPUTE QUERY VECTORS ====================
def get_tfidf_query_vector(query_text, tfidf_vectorizer):
    q_vec = tfidf_vectorizer.transform([query_text])
    q_vec = normalize(q_vec, norm='l2', axis=1)
    return q_vec

def get_sbert_query_vector(query_text, sbert_model):
    vec = sbert_model.encode(
        [query_text],
        normalize_embeddings=True,
        convert_to_numpy=True
    )[0].astype(np.float32)
    return vec


In [15]:
# ==================== HYBRID LATE FUSION (Dot Product) ====================
def hybrid_search(query_text, tfidf_vectorizer, D_tfidf, D_sbert, sbert_model, lambda_weight=0.7):
    # TF-IDF score
    q_tfidf = get_tfidf_query_vector(query_text, tfidf_vectorizer)
    tfidf_scores = (D_tfidf @ q_tfidf.T).toarray().flatten()

    # SBERT score
    q_sbert = get_sbert_query_vector(query_text, sbert_model)
    sbert_scores = D_sbert @ q_sbert

    # Late Fusion
    final_scores = (1-lambda_weight) * tfidf_scores + lambda_weight * sbert_scores
    return final_scores, tfidf_scores, sbert_scores


In [10]:
# ==================== EVALUATION ====================
def evaluate_model(queries, qrels_dict, doc_ids,
                   tfidf_vectorizer=None, D_tfidf=None,
                   D_sbert=None, sbert_model=None,
                   lambda_weight=0.7, k=10, name="Model"):

    print(f"\nĐang đánh giá {name}...")
    run_dict = {}
    eval_queries = {q: t for q, t in queries.items() if q in qrels_dict}

    for i, (q_id, q_text) in enumerate(eval_queries.items(), 1):
        if name == "TF-IDF Thuần":
            q_vec = get_tfidf_query_vector(q_text, tfidf_vectorizer)
            scores = (D_tfidf @ q_vec.T).toarray().flatten()

        elif name == "SBERT Thuần":
            q_vec = get_sbert_query_vector(q_text, sbert_model)
            scores = D_sbert @ q_vec

        else:  # Hybrid Late Fusion
            scores, _, _ = hybrid_search(
                query_text=q_text,
                tfidf_vectorizer=tfidf_vectorizer,
                D_tfidf=D_tfidf,
                D_sbert=D_sbert,
                sbert_model=sbert_model,
                lambda_weight=lambda_weight
            )

        top_k_idx = np.argsort(scores)[::-1][:k]
        run_dict[q_id] = {doc_ids[idx]: float(scores[idx]) for idx in top_k_idx}

        if i % 100 == 0 or i == len(eval_queries):
            print(f" → Đã xử lý {i:4d}/{len(eval_queries)} queries")

    evaluator = pytrec_eval.RelevanceEvaluator(
        qrels_dict,
        {"ndcg_cut_10", "recall_10", "recip_rank"}
    )
    results = evaluator.evaluate(run_dict)

    ndcg_scores = [v["ndcg_cut_10"] for v in results.values()]
    recall_scores = [v["recall_10"] for v in results.values()]
    mrr_scores = [v["recip_rank"] for v in results.values()]

    avg_ndcg = sum(ndcg_scores) / len(ndcg_scores)
    avg_recall = np.mean(recall_scores)
    avg_mrr = np.mean(mrr_scores)

    print(f"\n{'='*70}")
    print(f"KẾT QUẢ {name.upper()}")
    print(f"nDCG@10   : {avg_ndcg:.4f}")
    print(f"MRR@10    : {avg_mrr:.4f}")
    print(f"Recall@10 : {avg_recall:.4f}")
    print(f"{'='*70}")

    return avg_ndcg, avg_recall, avg_mrr


In [12]:
# ==================== CHẠY CHƯƠNG TRÌNH ====================
print("======= Load dữ liệu =======")
corpus = load_jsonl(CORPUS_PATH)
queries = load_jsonl(QUERIES_PATH)
qrels_dict = load_qrels(QRELS_PATH)

print(f"Corpus : {len(corpus):,} documents")
print(f"Queries: {len(queries):,} queries")
print(f"Qrels  : {len(qrels_dict):,} queries\n")

# 1. Build / Load TF-IDF
D_tfidf, doc_ids, tfidf_vectorizer = build_tfidf_version(corpus)

# 2. Build SBERT index
D_sbert, doc_ids_sbert = build_sbert_index(corpus, sbert_model, OUTPUT_DIR_SBERT, batch_size=SBERT_BATCH_SIZE)

assert doc_ids == doc_ids_sbert, "Lỗi: doc_ids của hai index không khớp!"


======= Load dữ liệu =======
Corpus : 3,633 documents
Queries: 3,237 queries
Qrels  : 323 queries


=== XÂY DỰNG TF-IDF THUẦN ===
TF-IDF Vocab size: 52,188 terms
Index TF-IDF đã lưu tại: /content/drive/MyDrive/InformationRetrieval/index_tfidf/

=== XÂY DỰNG SBERT INDEX ===
Đang encode 3,633 documents bằng SBERT (batch_size=64)...


Batches:   0%|          | 0/57 [00:00<?, ?it/s]

SBERT index đã lưu tại: /content/drive/MyDrive/InformationRetrieval/index_sbert/


In [16]:
# ====================== ĐÁNH GIÁ 3 MÔ HÌNH ======================
print("\n" + "="*85)
print("          BẮT ĐẦU ĐÁNH GIÁ SO SÁNH")
print("="*85)

results = {}

results["TF-IDF Thuần"] = evaluate_model(
    queries, qrels_dict, doc_ids,
    tfidf_vectorizer=tfidf_vectorizer, D_tfidf=D_tfidf,
    name="TF-IDF Thuần"
)

results["SBERT Thuần"] = evaluate_model(
    queries, qrels_dict, doc_ids,
    D_sbert=D_sbert, sbert_model=sbert_model,
    name="SBERT Thuần"
)

# Hybrid với nhiều lambda
lambdas = np.arange(0.5, 0.95, 0.05)
for lam in lambdas:
    name = f"Hybrid λ={lam:.2f}"
    ndcg, recall, mrr = evaluate_model(
        queries, qrels_dict, doc_ids,
        tfidf_vectorizer=tfidf_vectorizer, D_tfidf=D_tfidf,
        D_sbert=D_sbert, sbert_model=sbert_model,
        lambda_weight=lam,
        name=name
    )
    results[name] = (ndcg, recall, mrr)



          BẮT ĐẦU ĐÁNH GIÁ SO SÁNH

Đang đánh giá TF-IDF Thuần...
 → Đã xử lý  100/323 queries
 → Đã xử lý  200/323 queries
 → Đã xử lý  300/323 queries
 → Đã xử lý  323/323 queries

KẾT QUẢ TF-IDF THUẦN
nDCG@10   : 0.2485
MRR@10    : 0.4388
Recall@10 : 0.1110

Đang đánh giá SBERT Thuần...
 → Đã xử lý  100/323 queries
 → Đã xử lý  200/323 queries
 → Đã xử lý  300/323 queries
 → Đã xử lý  323/323 queries

KẾT QUẢ SBERT THUẦN
nDCG@10   : 0.2907
MRR@10    : 0.4886
Recall@10 : 0.1283

Đang đánh giá Hybrid λ=0.50...
 → Đã xử lý  100/323 queries
 → Đã xử lý  200/323 queries
 → Đã xử lý  300/323 queries
 → Đã xử lý  323/323 queries

KẾT QUẢ HYBRID Λ=0.50
nDCG@10   : 0.2880
MRR@10    : 0.4875
Recall@10 : 0.1360

Đang đánh giá Hybrid λ=0.55...
 → Đã xử lý  100/323 queries
 → Đã xử lý  200/323 queries
 → Đã xử lý  300/323 queries
 → Đã xử lý  323/323 queries

KẾT QUẢ HYBRID Λ=0.55
nDCG@10   : 0.2912
MRR@10    : 0.4891
Recall@10 : 0.1386

Đang đánh giá Hybrid λ=0.60...
 → Đã xử lý  100/323 queri

In [17]:
# ====================== BẢNG SO SÁNH ======================
print("\n" + "="*90)
print("                  BẢNG SO SÁNH KẾT QUẢ CUỐI CÙNG")
print("="*90)
print(f"{'Model':<35} {'nDCG@10':<10} {'Recall@10':<10} {'MRR@10':<10}")
print("-" * 70)
for model_name, (ndcg, recall, mrr) in results.items():
    print(f"{model_name:<35} {ndcg:<10.4f} {recall:<10.4f} {mrr:<10.4f}")
print("="*90)


                  BẢNG SO SÁNH KẾT QUẢ CUỐI CÙNG
Model                               nDCG@10    Recall@10  MRR@10    
----------------------------------------------------------------------
TF-IDF Thuần                        0.2485     0.1110     0.4388    
SBERT Thuần                         0.2907     0.1283     0.4886    
Hybrid λ=0.50                       0.2880     0.1360     0.4875    
Hybrid λ=0.55                       0.2912     0.1386     0.4891    
Hybrid λ=0.60                       0.2968     0.1399     0.5001    
Hybrid λ=0.65                       0.2999     0.1409     0.5020    
Hybrid λ=0.70                       0.3034     0.1420     0.5066    
Hybrid λ=0.75                       0.3072     0.1423     0.5144    
Hybrid λ=0.80                       0.3113     0.1430     0.5219    
Hybrid λ=0.85                       0.3148     0.1435     0.5302    
Hybrid λ=0.90                       0.3117     0.1396     0.5302    
